# 1. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Hiển thị toàn bộ các cột khi in
pd.set_option('display.max_columns', None)

# Đảm bảo thư mục output tồn tại để lưu ảnh báo cáo
os.makedirs('../output', exist_ok=True)

# 2. Đọc dữ liệu và kiểm tra tổng quan

In [ ]:
df = pd.read_csv('../data/hour.csv')
print(f"Kích thước dữ liệu: {df.shape}")
df.head()

In [ ]:
df.info()

# 3. Kiểm tra giá trị thiếu (Missing values)

In [ ]:
# Kiểm tra null trên từng cột
df.isnull().sum()

# 4. Thống kê mô tả và kiểm tra bất thường

In [ ]:
df.describe().T

Quan sát bảng thống kê mô tả, cột độ ẩm (`hum`) có giá trị nhỏ nhất `min = 0.0`. Trong điều kiện tự nhiên ngoài trời, độ ẩm không khí không thể bằng 0% được. Do đó, đây là các giá trị bất thường do cảm biến đo thời tiết bị lỗi ghi nhận.

In [ ]:
# Lọc các bản ghi có độ ẩm bằng 0
hum_zero = df[df['hum'] == 0]
print(f"Số dòng có độ ẩm bằng 0: {len(hum_zero)}")
hum_zero.head(10)

In [ ]:
# Kiểm tra phân bố theo ngày của các dòng lỗi
hum_zero['dteday'].value_counts()

Có 22 dòng bị lỗi độ ẩm bằng 0, tập trung chủ yếu liên tục vào ngày 2011-03-10. Các dòng này sẽ được xử lý ở bước tiền xử lý (cleaning).

# 5. Phân phối của biến mục tiêu (Histogram)

In [ ]:
# Vẽ phân phối của cnt gốc và log1p(cnt) (Histogram)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân phối gốc
sns.histplot(df['cnt'], kde=True, ax=axes[0], color='skyblue', bins=30)
mean_val = df['cnt'].mean()
median_val = df['cnt'].median()
axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
axes[0].axvline(median_val, color='green', linestyle='-', linewidth=2, label=f'Median: {median_val:.1f}')
axes[0].set_title('Phân phối lượng xe thuê gốc (Histogram)')
axes[0].set_xlabel('Lượng xe thuê (cnt)')
axes[0].set_ylabel('Tần suất')
axes[0].legend()

# Phân phối sau biến đổi log1p
log_cnt = np.log1p(df['cnt'])
sns.histplot(log_cnt, kde=True, ax=axes[1], color='salmon', bins=30)
axes[1].axvline(log_cnt.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {log_cnt.mean():.2f}')
axes[1].axvline(log_cnt.median(), color='green', linestyle='-', linewidth=2, label=f'Median: {log_cnt.median():.2f}')
axes[1].set_title('Phân phối sau khi biến đổi log1p (Histogram)')
axes[1].set_xlabel('log(cnt + 1)')
axes[1].set_ylabel('Tần suất')
axes[1].legend()

plt.tight_layout()
plt.savefig('../output/eda_1_histogram_log.png', dpi=300, bbox_inches='tight')
plt.show()

Biểu đồ cho thấy `cnt` bị lệch phải (right-skewed) rõ rệt với giá trị Mean (189.5) lớn hơn nhiều so với Median (142.0). Sau khi áp dụng biến đổi `log1p`, phân phối trở nên cân xứng và gần với phân phối chuẩn hơn, giúp giảm thiểu tác động của các giá trị quá lớn lên các mô hình hồi quy tuyến tính.

# 6. Phát hiện ngoại lai theo mùa (Boxplot)

In [ ]:
# Ánh xạ tên mùa để biểu đồ trực quan hơn
season_labels = {1: 'Mùa xuân', 2: 'Mùa hè', 3: 'Mùa thu', 4: 'Mùa đông'}
df['season_name'] = df['season'].map(season_labels)

plt.figure(figsize=(10, 5))
sns.boxplot(x='season_name', y='cnt', data=df, hue='season_name', palette='Set2', legend=False)
plt.title('Biểu đồ lượng xe thuê theo mùa (Boxplot)')
plt.xlabel('Mùa trong năm')
plt.ylabel('Lượng xe thuê (cnt)')

plt.savefig('../output/eda_2_boxplot_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Thống kê chính xác số lượng điểm ngoại lai theo từng mùa bằng phương pháp IQR
outlier_stats = []
for season in ['Mùa xuân', 'Mùa hè', 'Mùa thu', 'Mùa đông']:
    sub = df[df['season_name'] == season]['cnt']
    q1 = sub.quantile(0.25)
    median = sub.median()
    q3 = sub.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    n_outliers = (sub > upper_bound).sum()
    pct = (n_outliers / len(sub)) * 100
    outlier_stats.append({
        'Mùa': season,
        'Tổng số giờ': len(sub),
        'Ngưỡng râu trên': round(upper_bound, 1),
        'Số ngoại lai': n_outliers,
        'Tỷ lệ (%)': f'{pct:.2f}%'
    })

outlier_df = pd.DataFrame(outlier_stats)
outlier_df

Toàn bộ tập dữ liệu có tổng cộng 478 điểm ngoại lai (chiếm 2.75%). Trong đó, Mùa xuân (195 điểm) và Mùa đông (112 điểm) là hai mùa có nhiều điểm ngoại lai nhất do mặt bằng chung thấp khiến ngưỡng râu trên bị kéo xuống. Mùa đông cũng ghi nhận lượng thuê đỉnh điểm lên tới 967 xe/giờ, ngang ngửa Mùa hè (968 xe) và Mùa thu (977 xe). Các điểm ngoại lai này sẽ làm lệch trọng số mô hình và gây hiện tượng Overfitting cho các thuật toán nhạy nhiễu như AdaBoost, do đó cần được xử lý ở bước tiền xử lý dữ liệu.

# 7. Phân tích xu hướng theo 24 giờ (Fever Chart / Line Chart)

In [ ]:
# Xu hướng thuê xe trung bình theo 24 giờ trong ngày (Fever chart / Line chart)
plt.figure(figsize=(12, 5))
working_df = df.groupby(['hr', 'workingday'])['cnt'].mean().reset_index()
working_df['workingday_name'] = working_df['workingday'].map({
    1: 'Ngày đi làm (Workingday)', 
    0: 'Ngày nghỉ / Cuối tuần (Weekend/Holiday)'
})

sns.lineplot(x='hr', y='cnt', hue='workingday_name', data=working_df, marker='o', linewidth=2.5, palette=['#e74c3c', '#3498db'])
plt.title('Xu hướng thuê xe trung bình theo 24 giờ (Fever Chart / Line Chart)')
plt.xlabel('Khung giờ trong ngày (0h - 23h)')
plt.ylabel('Lượng xe thuê trung bình (cnt)')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='')
plt.savefig('../output/eda_3_time_trends.png', dpi=300, bbox_inches='tight')
plt.show()

Biểu đồ chỉ ra sự khác biệt rõ rệt về hành vi thuê xe giữa ngày làm việc và ngày nghỉ:
- Ngày làm việc (đường màu xanh): Xuất hiện 2 đỉnh cao điểm rất nhọn (Fever peaks) vào lúc 8h sáng (giờ đi làm) và 17h-18h chiều (giờ tan tầm).
- Ngày nghỉ / Cuối tuần (đường màu đỏ): Chỉ có 1 đỉnh thoai thoải vào khoảng 12h-16h trưa phục vụ mục đích dạo chơi, giải trí.
Sự khác biệt rõ ràng này là căn cứ thực nghiệm quan trọng để ta tạo thêm đặc trưng mới `is_rush_hour` ở bước kỹ thuật đặc trưng (feature engineering).

# 8. Kiểm tra đa cộng tuyến (Scatter Plot & Correlation Heatmap)

In [ ]:
# Scatter plot và Correlation Heatmap kiểm tra đa cộng tuyến
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Quan hệ giữa temp và atemp (Scatter Plot)
corr_val = df['temp'].corr(df['atemp'])
sns.regplot(x='temp', y='atemp', data=df, ax=axes[0], color='#2c3e50', 
            scatter_kws={'alpha': 0.15, 's': 15}, line_kws={'color': '#e74c3c', 'linewidth': 2})
axes[0].set_title(f'Quan hệ giữa temp và atemp (Scatter Plot)\n(Tương quan Pearson: {corr_val:.4f})')
axes[0].set_xlabel('Nhiệt độ thực tế (temp)')
axes[0].set_ylabel('Nhiệt độ cảm nhận (atemp)')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Subplot 2: Correlation Heatmap
numeric_cols = ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, 
            square=True, linewidths=0.5, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Ma trận tương quan giữa các đặc trưng (Correlation Heatmap)')

plt.tight_layout()
plt.savefig('../output/eda_4_scatter_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

Hệ số tương quan giữa nhiệt độ thực tế (`temp`) và nhiệt độ cảm nhận (`atemp`) lên tới **0.99**, các điểm trên Scatter plot xếp thành một đường thẳng hoàn hảo. Đây là hiện tượng **Đa cộng tuyến (Multicollinearity)** nghiêm trọng. Trong mô hình hồi quy tuyến tính, đa cộng tuyến sẽ làm ma trận $(X^T X)$ gần suy biến và khiến phương sai của các trọng số hồi quy tăng vọt, gây mất ổn định mô hình. Do đó, cột `atemp` bắt buộc phải bị loại bỏ ở bước tiền xử lý dữ liệu.
Ngoài ra, ma trận tương quan cũng chứng minh $cnt = casual + registered$ với mối tương quan rất cao, khẳng định nguy cơ rò rỉ dữ liệu (Data Leakage) nếu không xóa 2 cột này trước khi huấn luyện mô hình.